# 🚀 Google Colab (1x GPU T4) - SigLIP 2 SO400M-384 Keyframe Indexing Pipeline
### 🎯 Mục tiêu: Trích xuất 177,321 Keyframes bằng `google/siglip2-so400m-patch14-384` trên Google Colab T4 GPU
- **Model**: `google/siglip2-so400m-patch14-384` (Embedding dim: **1152**)
- **Độ phân giải**: **`384 x 384`** (Patch 14x14 = 729 visual tokens quan sát chi tiết vi mô)
- **Tối ưu Colab T4**: Single GPU FP16 Autocast + Batch Size 64 mượt mà không tràn VRAM
- **Tính năng an toàn**: Tự động tải lại khi lỗi mạng + Kiểm tra đủ 100% 177,321 ảnh trước khi index
- **Đầu ra**: Tự động lưu thẳng vào **Google Drive** (`/content/drive/MyDrive/embeddings_siglip2_384.npy`) và kích hoạt tải về máy.

In [ ]:
# 1. Kết nối Google Drive & Cài đặt thư viện cần thiết (Không đè Pillow/Torchvision để tránh lỗi kernel)
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
!apt-get update -qq && apt-get install -y -qq aria2
!pip install -q -U transformers accelerate


In [ ]:
# 2. Tải toàn bộ 15 file ZIP Keyframes & Metadata chính thức từ BTC (Tự động tải lại nếu lỗi mạng)
import os
import time
import zipfile
import subprocess
import urllib.request

ZIP_DIR = "/content/zips"
os.makedirs(ZIP_DIR, exist_ok=True)

# DANH SÁCH 15 LINK CHÍNH THỨC CỦA BAN TỔ CHỨC
ZIP_URLS = [
    "https://aic-data.ledo.io.vn/map-keyframes-aic25-b1.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L21.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L22.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L23.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L24.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L25.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_a.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_b.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_c.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_d.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_e.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L27.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L28.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L29.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L30.zip"
]

def is_zip_valid(file_path):
    if not os.path.exists(file_path) or os.path.getsize(file_path) < 1000:
        return False
    try:
        with zipfile.ZipFile(file_path, 'r') as zf:
            first_bad = zf.testzip()
            return first_bad is None
    except Exception:
        return False

def download_all_zips_with_retry(urls, max_retries=10):
    for attempt in range(1, max_retries + 1):
        missing_urls = []
        for url in urls:
            fname = os.path.basename(url)
            fpath = os.path.join(ZIP_DIR, fname)
            if not is_zip_valid(fpath):
                missing_urls.append(url)
                
        if not missing_urls:
            print(f"\n🎉 XÁC NHẬN TOÀN BỘ {len(urls)}/{len(urls)} FILE ZIP HỢP LỆ & NGUYÊN VẸN 100%!")
            break
            
        print(f"\n🔄 [Lần thử {attempt}/{max_retries}] Đang tải {len(missing_urls)} file ZIP chưa hoàn chỉnh:")
        for u in missing_urls:
            print(f"   - {os.path.basename(u)}")
            
        urls_file = "/content/missing_urls.txt"
        with open(urls_file, "w") as f:
            for u in missing_urls:
                f.write(u + "\n")
                
        cmd = [
            "aria2c", "-i", urls_file, "-d", ZIP_DIR, 
            "-j", "6", "-x", "16", "-s", "16", "-k", "1M",
            "--allow-overwrite=true", "--auto-file-renaming=false",
            "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            "--summary-interval=5"
        ]
        subprocess.run(cmd)
        time.sleep(2)

download_all_zips_with_retry(ZIP_URLS)


In [ ]:
# 3. Giải nén toàn bộ Keyframes ra /content/keyframes/
import os
import glob

KEYFRAMES_DIR = "/content/keyframes"
os.makedirs(KEYFRAMES_DIR, exist_ok=True)

zips = sorted(glob.glob("/content/zips/Keyframes_*.zip"))
print(f"📦 Tìm thấy {len(zips)} file ZIP keyframes (L21 đến L30). Đang giải nén...")

for zp in zips:
    fname = os.path.basename(zp)
    print(f"  -> Đang giải nén {fname} ({os.path.getsize(zp)/(1024*1024):.1f} MB)...")
    !unzip -q -o "{zp}" -d {KEYFRAMES_DIR}/

print("✅ Giải nén toàn bộ keyframes thành công!")


In [ ]:
# 4. KIỂM TRA ĐẦY ĐỦ 100% SỐ LƯỢNG 177,321 ẢNH THEO METADATA CHUẨN CỦA BTC
import os
import io
import csv
import zipfile
import urllib.request
from PIL import Image
from collections import defaultdict

KEYFRAMES_DIR = "/content/keyframes"
MAP_KEYFRAMES_PATH = "/content/zips/map-keyframes-aic25-b1.zip"

print("🔍 Đang quét cấu trúc thư mục keyframes trên ổ cứng Colab...")
video_map = {}
for root, dirs, files in os.walk(KEYFRAMES_DIR):
    for d in dirs:
        if d.startswith("L") and "_V" in d:
            video_map[d] = os.path.join(root, d)
            video_map[d.upper()] = os.path.join(root, d)
            video_map[d.lower()] = os.path.join(root, d)

print(f"✅ Đã định vị {len(video_map) // 3} thư mục video trên đĩa.")

print("📋 Đang phân tích metadata chuẩn từ map-keyframes-aic25-b1.zip...")
metadata_items = []
batch_counts = defaultdict(int)

with zipfile.ZipFile(MAP_KEYFRAMES_PATH, 'r') as z:
    csv_files = sorted([f for f in z.namelist() if f.endswith('.csv')])
    for csv_path in csv_files:
        video_id = os.path.splitext(os.path.basename(csv_path))[0]
        batch_name = video_id.split("_")[0] if "_" in video_id else "Khác"
        content = z.read(csv_path).decode('utf-8')
        reader = csv.DictReader(content.splitlines())
        for i, row in enumerate(reader):
            n = int(row.get('n', i + 1))
            actual_frame_idx = int(float(row.get('frame_idx', 0)))
            metadata_items.append({
                'video_id': video_id,
                'n': n,
                'frame_filename': f'{n:03d}.jpg',
                'frame_idx': actual_frame_idx
            })
            batch_counts[batch_name] += 1

total_expected = len(metadata_items)
print(f"📦 Tổng số khung hình BTC yêu cầu: {total_expected:,} frames.")
print("📊 Phân bổ khung hình theo từng Batch:")
for b_name, b_cnt in sorted(batch_counts.items()):
    print(f"   - {b_name}: {b_cnt:,} frames")

print("\n⚡ Đang kiểm tra tính hợp lệ của từng file ảnh trên ổ cứng...")
missing_files = []
corrupted_files = []

for idx, item in enumerate(metadata_items):
    v_id = item['video_id']
    f_name = item['frame_filename']
    
    if v_id not in video_map:
        missing_files.append(f"{v_id}/{f_name} (Thư mục video không tồn tại)")
        continue
        
    img_path = os.path.join(video_map[v_id], f_name)
    if not os.path.exists(img_path):
        missing_files.append(img_path)
    elif idx % 2000 == 0: # Kiểm tra mẫu ngẫu nhiên mở file ảnh
        try:
            with Image.open(img_path) as img:
                img.verify()
        except Exception:
            corrupted_files.append(img_path)

print("="*60)
if len(missing_files) == 0 and len(corrupted_files) == 0:
    print(f"🎉 HOÀN HẢO 100%! ĐÃ XÁC NHẬN TOÀN BỘ {total_expected:,}/{total_expected:,} FILE ẢNH ĐẦY ĐỦ VÀ HỢP LỆ!")
    print("👉 SẴN SÀNG CHUYỂN SANG BƯỚC 5 ĐỂ TRÍCH XUẤT VECTOR TRÊN GPU T4!")
else:
    print(f"❌ CẢNH BÁO: Phát hiện {len(missing_files)} file bị thiếu và {len(corrupted_files)} file bị lỗi!")
    if missing_files:
        print("Mẫu file bị thiếu:", missing_files[:5])
    raise RuntimeError("Dữ liệu chưa đầy đủ 100%. Vui lòng chạy lại Bước 2 và Bước 3!")
print("="*60)


In [ ]:
# 5. Trích xuất Embeddings SigLIP 2 SO400M-384 trên 1x GPU T4 (FP16 Autocast)
import os
import io
import gc
import csv
import time
import zipfile
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModel
from google.colab import files

MODEL_NAME = "google/siglip2-so400m-patch14-384"
DRIVE_OUTPUT_PATH = "/content/drive/MyDrive/embeddings_siglip2_384.npy"
LOCAL_OUTPUT_PATH = "/content/embeddings_siglip2_384.npy"
TMP_MEMMAP_PATH = "/content/embeddings_siglip2_384_tmp.dat"
KEYFRAMES_DIR = "/content/keyframes"
MAP_KEYFRAMES_PATH = "/content/zips/map-keyframes-aic25-b1.zip"

# 🔥 CẤU HÌNH TỐI ƯU CHO 1x GPU TESLA T4 (16GB VRAM):
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64     # 64 ảnh 384x384 / batch (chiếm ~8.5GB VRAM ổn định 100%)
NUM_WORKERS = 2     # 2 tiến trình nạp ảnh nền song song trên Colab CPU
EMBED_DIM = 1152    # SigLIP 2 SO400M output dimension = 1152

print(f"🚀 Thiết bị tính toán: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
if torch.cuda.is_available():
    print(f"🔥 VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    torch.backends.cudnn.benchmark = True

# Tự động nạp lại map và metadata nếu chưa có sẵn trong memory
if 'video_map' not in globals() or 'metadata_items' not in globals():
    print("🔍 Đang ánh xạ video và metadata...")
    video_map = {}
    for root, dirs, files_list in os.walk(KEYFRAMES_DIR):
        for d in dirs:
            if d.startswith("L") and "_V" in d:
                video_map[d] = os.path.join(root, d)
                video_map[d.upper()] = os.path.join(root, d)
                video_map[d.lower()] = os.path.join(root, d)
                
    metadata_items = []
    with zipfile.ZipFile(MAP_KEYFRAMES_PATH, 'r') as z:
        csv_files = sorted([f for f in z.namelist() if f.endswith('.csv')])
        for csv_path in csv_files:
            video_id = os.path.splitext(os.path.basename(csv_path))[0]
            content = z.read(csv_path).decode('utf-8')
            reader = csv.DictReader(content.splitlines())
            for i, row in enumerate(reader):
                n = int(row.get('n', i + 1))
                actual_frame_idx = int(float(row.get('frame_idx', 0)))
                metadata_items.append({
                    'video_id': video_id,
                    'n': n,
                    'frame_filename': f'{n:03d}.jpg',
                    'frame_idx': actual_frame_idx
                })

# Dataset
class KeyframeDataset(Dataset):
    def __init__(self, metadata, v_map, processor):
        self.metadata = metadata
        self.video_map = v_map
        self.processor = processor

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        item = self.metadata[idx]
        v_id = item["video_id"]
        f_name = item["frame_filename"]
        img_path = os.path.join(self.video_map[v_id], f_name)
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (384, 384), (0, 0, 0))
        return img

def collate_fn(batch, processor):
    return processor(images=batch, return_tensors="pt")["pixel_values"]

def run_colab_t4_indexing():
    total_count = len(metadata_items)
    print(f"📦 Tổng số keyframes trích xuất: {total_count:,}")
    
    print(f"🤗 Đang nạp mô hình: {MODEL_NAME}...")
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(device)
    model.eval()
    
    dataset = KeyframeDataset(metadata_items, video_map, processor)
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        collate_fn=lambda b: collate_fn(b, processor)
    )
    
    if os.path.exists(TMP_MEMMAP_PATH):
        os.remove(TMP_MEMMAP_PATH)
    memmap_matrix = np.memmap(TMP_MEMMAP_PATH, dtype='float32', mode='w+', shape=(total_count, EMBED_DIM))
    
    processed = 0
    start_time = time.time()
    print(f"🚀 Bắt đầu trích xuất trên Colab T4 GPU (Batch: {BATCH_SIZE}, FP16 Autocast)...\n")
    
    with torch.inference_mode():
        for batch_idx, pixel_values in enumerate(dataloader):
            pixel_values = pixel_values.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda"), dtype=torch.float16):
                outputs = model.get_image_features(pixel_values=pixel_values)
                vecs = outputs.pooler_output if hasattr(outputs, "pooler_output") else (outputs.image_embeds if hasattr(outputs, "image_embeds") else outputs[0])
                vecs = vecs / vecs.norm(dim=-1, keepdim=True)
                
            feats_np = vecs.cpu().to(torch.float32).numpy()
            bsz = len(feats_np)
            memmap_matrix[processed : processed + bsz] = feats_np
            processed += bsz
            
            if (batch_idx + 1) % 25 == 0 or processed == total_count:
                memmap_matrix.flush()
                elapsed = time.time() - start_time
                fps = processed / elapsed if elapsed > 0 else 0
                eta_min = ((total_count - processed) / fps) / 60 if fps > 0 else 0
                percent = (processed / total_count) * 100.0
                print(f"🔥 Tiến độ: {processed:,}/{total_count:,} ({percent:.1f}%) | Tốc độ: {fps:.1f} fps | Đã chạy: {elapsed/60:.1f}m | Còn lại: {eta_min:.1f}m")
                
    memmap_matrix.flush()
    total_time = time.time() - start_time
    print(f"\n🎉 HOÀN THÀNH TRÍCH XUẤT 100%! Thời gian chạy: {total_time/60:.2f} phút!")
    
    final_matrix = np.array(memmap_matrix)
    
    # 1. Lưu bản sao cục bộ trên Colab
    print(f"💾 Đang lưu bản sao cục bộ tại: {LOCAL_OUTPUT_PATH}...")
    np.save(LOCAL_OUTPUT_PATH, final_matrix)
    
    # 2. Lưu an toàn vĩnh viễn vào Google Drive (Chắc chắn 100% không mất)
    try:
        print(f"☁️ Đang lưu an toàn vào Google Drive của bạn: {DRIVE_OUTPUT_PATH}...")
        np.save(DRIVE_OUTPUT_PATH, final_matrix)
        print("✅ ĐÃ ĐỒNG BỘ THÀNH CÔNG VÀO GOOGLE DRIVE!")
    except Exception as e:
        print(f"⚠️ Không thể lưu trực tiếp vào Drive ({e}). File vẫn còn tại {LOCAL_OUTPUT_PATH}")
        
    print(f"🎉 Kích thước ma trận hoàn tất: {final_matrix.shape} ({os.path.getsize(LOCAL_OUTPUT_PATH)/(1024*1024):.2f} MB)")
    
    del memmap_matrix
    if os.path.exists(TMP_MEMMAP_PATH):
        os.remove(TMP_MEMMAP_PATH)
        
    # 3. Kích hoạt tải trực tiếp về máy qua trình duyệt
    try:
        print("🚀 Đang kích hoạt tải file về máy tính cá nhân của bạn qua trình duyệt...")
        files.download(LOCAL_OUTPUT_PATH)
    except Exception as e:
        print(f"ℹ️ Nếu trình duyệt chặn popup tải tự động, bạn có thể lấy file tại Google Drive hoặc chạy Cell 6!")

if __name__ == "__main__":
    run_colab_t4_indexing()


In [ ]:
# 6. Tải lại file về máy thủ công (Nếu bạn vô tình đóng popup ở Cell 5)
from google.colab import files
import os

file_path = "/content/embeddings_siglip2_384.npy"
if os.path.exists(file_path):
    print(f"🚀 Đang tải file {file_path} ({os.path.getsize(file_path)/(1024*1024):.2f} MB) về máy...")
    files.download(file_path)
else:
    print("❌ File chưa được tạo. Hãy chạy Cell 5 trước!")
